In [2]:
import ee
import pandas as pd


In [3]:
ee.Authenticate()

True

In [4]:
# Initialize Earth Engine
ee.Initialize()

EEException: ee.Initialize: no project found. Call with project= or see http://goo.gle/ee-auth.

In [ ]:
# ====== 0) Parameters ======
SLOPE_MAX_DEG = 5
WDPA_STATUSES = ['Designated', 'Inscribed', 'Established']
WDPA_PADDING_M = 100
BUFFER_METERS = 1000
QUICK_SCALE = 150
M2_PER_KM2 = 1e6
M2_PER_ACRE = 4046.8564224
ACRES_PER_MW = 4.0

# Colors
# PALETTE_SOLAR = ['#FFFF4C', '#B4B4B4'] # Grassland, Bare


In [9]:
# Load your points into Earth Engine 
# (It's best to upload your CSV as an 'Asset' in the GEE Console first)
points = ee.FeatureCollection('projects/ee-land-availability-iecc/assets/facility_country_sector_subsector')

In [10]:
# ====== Inputs ======
wc = ee.Image('ESA/WorldCover/v200/2021').select('Map')
elev = ee.Image('USGS/SRTMGL1_003')
wdpa = ee.FeatureCollection('WCMC/WDPA/current/polygons') \
    .filter(ee.Filter.inList('STATUS', WDPA_STATUSES))

In [11]:

# Slope Mask: Slope must be <= 5 degrees
slopeMask = ee.Terrain.slope(elev).lte(SLOPE_MAX_DEG)

# Protected Areas Mask: Paint WDPA polygons and add a buffer
wdpaMask = ee.Image().paint(wdpa, 1).unmask(0) \
    .focal_max(radius=WDPA_PADDING_M, units='meters')

# Final Masks: Inverse of protected areas (where value is 0)
protectedMask = wdpaMask.eq(0)

# Combined Suitability: Land Cover AND Slope AND Protected Status
suitableMask = slopeMask.And(protectedMask)


In [12]:
# 1. Define the class name mapping (Python Dictionary)
class_names = {
    '10': 'tree_cover_km2',
    '20': 'shrubland_km2',
    '30': 'grassland_km2',
    '40': 'cropland_km2',
    '50': 'built_up_km2', # Includes buildings, roads, urban parks, etc
    '60': 'bare_sparse_km2',
    '70': 'snow_ice_km2',
    '80': 'water_km2',
    '90': 'wetland_km2',
    '95': 'mangroves_km2',
    '100': 'moss_lichen_km2'
}

### Binary mask

In [ ]:
# # ====== Suitability masks ======

# # Land Cover Mask: Grassland (30) or Bare/Sparse (60)
# lcMask = wc.eq(30).Or(wc.eq(60)) 

# # Area Calculation: Calculate pixel area where suitable
# suitableAreaImage = ee.Image.pixelArea().updateMask(suitableMask)

# # 4. Map the buffer and calculation over the entire collection
# def calculate_area(feature):
#     buffer = feature.geometry().buffer(BUFFER_METERS)
#     # reduceRegion inside a map is the 'server-side' way to loop
#     stats = suitableAreaImage.reduceRegion(
#         reducer=ee.Reducer.sum(),
#         geometry=buffer,
#         scale=30, # Consider 30m for 45k points to speed up processing
#         maxPixels=1e13
#     )
#     return feature.set('suitable_km2', stats.get('area'))

# results = points.map(calculate_area)


### Categorized mask

In [13]:
# Convert the Python dict to an ee.Dictionary for server-side use
ee_class_names = ee.Dictionary(class_names)

# 2. Prepare the Image Stack
# Combined suitability (excluding LC) to mask the area calculation
environmental_mask = slopeMask.And(protectedMask)

# Stack Pixel Area and WorldCover
# Band 0: 'area', Band 1: 'lc'
area_by_lc_image = ee.Image.pixelArea() \
    .updateMask(environmental_mask) \
    .addBands(wc) \
    .rename(['area', 'lc'])

In [14]:
def calculate_grouped_area(feature):
    buffer = feature.geometry().buffer(BUFFER_METERS)
    
    stats = area_by_lc_image.reduceRegion(
        reducer=ee.Reducer.sum().group(
            groupField=1,
            groupName='class'
        ),
        geometry=buffer,
        scale=30,
        maxPixels=1e13
    )
    
    stats_list = ee.List(stats.get('groups'))
    
    def rename_columns(group, d):
        g = ee.Dictionary(group)
        class_val = g.getNumber('class').format()
        area_km2 = g.getNumber('sum').divide(1e6)
        
        # CHANGED: Use .cat() instead of .concat() 
        # Also added a check to ensure we get a string key
        default_name = ee.String('class_').cat(class_val)
        name = ee.String(ee_class_names.get(class_val, default_name))
        
        return ee.Dictionary(d).set(name, area_km2)
    
    land_cover_results = ee.Dictionary(stats_list.iterate(rename_columns, ee.Dictionary({})))
    
    return feature.set(land_cover_results)

### Calculate area of available land

In [15]:
# 3. Apply the processing
results = points.map(calculate_grouped_area)

id_columns = ['source_name', 'source_id']
selector_list = id_columns + ['system:index'] + list(class_names.values())

task = ee.batch.Export.table.toDrive(
    collection=results,
    description=f'LandCover_Area_Categorized_{BUFFER_METERS//1000}km',
    fileFormat='CSV',
    selectors=selector_list
)

# Start the task
task.start()
print("Task started. Check your GEE Task Manager or Google Drive for the results.")

Task started. Check your GEE Task Manager or Google Drive for the results.


In [ ]:
# # 5. Execute Geospatial Batch Export
# task = ee.batch.Export.table.toDrive(
#     collection=results,
#     description='Solar_Suitability_Geospatial',
#     fileFormat='GeoJSON',
#     folder='GEE_Exports',
#     selectors=['source_id', 'source_name', 'suitable_km2', 'solar_potential_MW', '.geo']
# )

# task.start()
# print("Geospatial export task started. Monitor progress in the GEE Tasks tab.")

Geospatial export task started. Monitor progress in the GEE Tasks tab.


### Generate raster data of available land

In [ ]:
# Create the buffer geometry for all points (union them for efficiency)
# Note: For 45k points, a single union might be heavy; 
# GEE handles spatial intersections well even without an explicit union.
buffered_points = points.map(lambda f: f.buffer(BUFFER_METERS))

# Clip the suitableAreaImage to the buffered areas
# This creates a raster that only exists within 15km of your facilities
final_raster = suitableAreaImage.clip(buffered_points)

# Optional: Convert to a 1/0 binary mask (1 = Suitable, 0 = Unsuitable/Buffer)
# This makes the output much smaller and easier to use in GIS
export_image = final_raster.gt(0).unmask(0).clip(buffered_points).uint8()

In [11]:
# 4. Execute Raster Batch Export
# Note: 'region' should be the bounding box of your points or the buffered collection
task = ee.batch.Export.image.toDrive(
    image=export_image,
    description='Facility_Solar_Suitability_Raster',
    folder='GEE_Exports',
    fileNamePrefix='suitable_land_15km',
    scale=30, # 30m resolution matches your landcover data
    region=buffered_points.geometry().bounds(), 
    maxPixels=1e13,
    crs='EPSG:4326'
)

task.start()
print("Raster export task started. Check the GEE Tasks tab.")

Raster export task started. Check the GEE Tasks tab.


### Explore Land cover area

In [18]:
land_cover_df = pd.read_csv('LandCover_Area_Categorized_5km.csv')

In [19]:
land_cover_df.head(n=50)

,source_name,source_id,system:index,tree_cover_km2,shrubland_km2,grassland_km2,cropland_km2,built_up_km2,bare_sparse_km2,snow_ice_km2,water_km2,wetland_km2,mangroves_km2,moss_lichen_km2
0,Abu Dhabi Municipal Region,3672939,00000000000000002a25,0.201998,0.037551,0.043697,0.016329,33.602665,17.839566,NaN,6.347310,0.000179,NaN,NaN
1,Taweelah Shaheen Alumina Refinery,43765087,000000000000000042e2,0.190701,0.179141,0.478068,0.155185,6.778201,34.578052,NaN,21.970526,0.028443,0.957301,NaN
2,Jebel Ali aluminium plant,3672938,000000000000000037e4,0.694474,0.230654,0.064953,0.017051,19.135908,14.228459,NaN,26.412051,NaN,NaN,NaN
3,Taweelah-Abu Dhabi aluminium plant,3672937,00000000000000008f08,0.684053,0.317069,0.529765,0.380762,5.841910,29.568469,NaN,23.980467,0.058552,2.436477,NaN
4,Puerto Madryn aluminium plant,3672940,000000000000000076bd,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,NaN,NaN,NaN
5,Bunbury City,3672948,000000000000000000b1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,77.566482,NaN,NaN,NaN
6,Gladstone Region,3672950,0000000000000000062d,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN
7,Worsley Alumina Refinery,43765099,00000000000000003d89,44.035103,0.009009,1.571964,0.041294,0.301045,7.527393,NaN,1.326653,NaN,NaN,NaN
8,Collie Shire,3672949,0000000000000000537f,41.778912,0.001500,4.292982,0.697518,0.074126,0.131493,NaN,0.012755,NaN,NaN,NaN
9,Kwinana Alumina Refinery,43765020,0000000000000000613c,6.524343,0.164048,8.167236,0.651428,6.827805,2.927602,NaN,30.748403,0.037984,NaN,NaN
